# GBM calibration & Monte Carlo — period 2018–2019

**Period file:** one notebook = one sample window. This file covers **2018-01-01 → 2019-12-31**.

| Role | Ticker |
|------|--------|
| Primary | **SPY** |
| Secondary | AAPL |
| Secondary | MSFT |

**Contents**
1. Historical adjusted-close trends for the period
2. Strike prices observed in the options panels for this window
3. GBM parameter estimation formulas
4. Daily rolling calibration (1-year / 252-trading-day lookback)
5. Interactive Monte Carlo (fixed parameters, **Start / Restart**, no sliders) — **six graphs** (2 per ticker)


## 0. Setup


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

DATA = Path("..") / "data"
PERIOD_START = pd.Timestamp("2018-01-01")
PERIOD_END = pd.Timestamp("2019-12-31")
TICKERS = ["AAPL", "MSFT", "SPY"]  # SPY is primary
N_DAYS = 252
LOOKBACK_DAYS = 252  # 1-year calibration window (trading days)
COLORS = {"AAPL": "#1f77b4", "MSFT": "#ff7f0e", "SPY": "#2ca02c"}

prices = pd.read_csv(DATA / "equity" / "prices_clean.csv", parse_dates=["Date"]).set_index("Date").sort_index()
period_prices = prices.loc[PERIOD_START:PERIOD_END, TICKERS].copy()
log_returns_all = np.log(prices[TICKERS]).diff()

print(f"Price sample: {prices.index.min().date()} → {prices.index.max().date()}")
print(
    f"Period rows: {len(period_prices)} trading days "
    f"({period_prices.index.min().date()} → {period_prices.index.max().date()})"
)
period_prices.head()


## 1. Stock price trends (2018–2019)

Adjusted close for AAPL, MSFT, and SPY (primary) over the period.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

for ax, ticker in zip(axes, TICKERS):
    s = period_prices[ticker].dropna()
    ax.plot(s.index, s.values, color=COLORS[ticker], lw=1.4)
    role = "primary" if ticker == "SPY" else "secondary"
    ax.set_ylabel("Adj close")
    ax.set_title(f"{ticker} ({role}) — adjusted close, 2018–2019")

axes[-1].set_xlabel("Date")
fig.suptitle("Stock price trends — period 2018–2019", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

display(period_prices.describe().T[["count", "mean", "min", "max"]].round(4))


## 2. Strike prices in this period

Unique strikes \(K\) from the processed American options panels (`*_options_panel.csv`), filtered to quotes with `trading_date` inside **2018-01-01 → 2019-12-31**.

> **Note:** AAPL strikes are on the **option/contract scale** (pre-split style), while equity adj closes are split-adjusted — do not mix them without a scale check.


In [ ]:
for ticker in TICKERS:
    path = DATA / "options" / "processed" / f"{ticker}_options_panel.csv"
    opt = pd.read_csv(path, usecols=["trading_date", "K"], parse_dates=["trading_date"])
    m = (opt["trading_date"] >= PERIOD_START) & (opt["trading_date"] <= PERIOD_END)
    sub = opt.loc[m]
    uniq = np.sort(sub["K"].dropna().unique())
    dmin, dmax = sub["trading_date"].min(), sub["trading_date"].max()
    display(Markdown(
        f"### {ticker} — {len(uniq)} unique strikes "
        f"(options quotes {dmin.date() if pd.notna(dmin) else 'n/a'} → "
        f"{dmax.date() if pd.notna(dmax) else 'n/a'})"
    ))
    print("Strikes K:", ", ".join(f"{x:g}" for x in uniq))
    display(pd.DataFrame({"K": uniq}).T)


## 3. Estimation formulas (GBM)

Under GBM,

$$dS_t = \mu S_t\, dt + \sigma S_t\, dW_t$$

with discrete simulation

$$S_{t+\Delta t} = S_t \exp\Big(\big(\mu - \tfrac{1}{2}\sigma^2\big)\Delta t + \sigma\sqrt{\Delta t}\, Z\Big),\quad Z\sim N(0,1).$$

### Parameters to estimate

| Parameter | Meaning | Estimator (daily data, \(N=252\)) |
|-----------|---------|-------------------------------------|
| \(\mu\) | annual drift | \(\hat\mu = \bar r \times 252\), \(\bar r = \frac{1}{n}\sum r_t\) |
| \(\sigma\) | annual volatility | \(\hat\sigma = s_r \times \sqrt{252}\), \(s_r = \sqrt{\frac{1}{n-1}\sum(r_t-\bar r)^2}\) |
| \(S_0\) | start price | first adjusted close in the forecast period |
| \(r_t\) | log return | \(r_t = \ln(S_t / S_{t-1})\) |

### Daily rolling calibration (this notebook)

For each trading day \(t\) in 2018-01-01 … 2019-12-31:

1. Take the lookback window of the previous **252 trading days** (≈ 1 year) ending at \(t\).
2. Collect daily log returns in that window.
3. Compute \(\hat\mu(t)\) and \(\hat\sigma(t)\) with the formulas above.

**Monte Carlo below uses fixed parameters:** the rolling estimates on the **first trading day of the period** — held constant for the whole 2018–2019 simulation.


## 4. Daily rolling calibration (1-year / 252-day window) — 2018–2019


In [ ]:
def daily_rolling_calibration(ticker: str) -> pd.DataFrame:
    """Daily rolling μ̂, σ̂ with a trailing LOOKBACK_DAYS (252 ≈ 1y) window."""
    rets = log_returns_all[ticker].dropna()
    roll_mu = rets.rolling(LOOKBACK_DAYS).mean() * N_DAYS
    roll_sigma = rets.rolling(LOOKBACK_DAYS).std(ddof=1) * np.sqrt(N_DAYS)
    # window_start = date of the first return inside the 252-day window ending at t
    window_start = rets.index.to_series().shift(LOOKBACK_DAYS - 1)

    out = pd.DataFrame({
        "date": rets.index,
        "window_start": window_start.values,
        "window_end": rets.index,
        "n_days": LOOKBACK_DAYS,
        "mu": roll_mu.values,
        "sigma": roll_sigma.values,
    }).dropna(subset=["mu", "sigma"])
    out = out.loc[(out["date"] >= PERIOD_START) & (out["date"] <= PERIOD_END)].reset_index(drop=True)
    return out


rolling = {t: daily_rolling_calibration(t) for t in TICKERS}

mc_params = {}
for t in TICKERS:
    row0 = rolling[t].iloc[0]
    hist = period_prices[t].dropna()
    s0 = float(hist.iloc[0])
    t0, t1 = hist.index[0], hist.index[-1]
    T_years = (t1 - t0).days / 365.25
    n_steps = int(len(hist) - 1)
    mc_params[t] = {
        "mu": float(row0["mu"]),
        "sigma": float(row0["sigma"]),
        "S0": s0,
        "t0": t0,
        "t1": t1,
        "T_years": float(T_years),
        "n_steps": n_steps,
        "cal_date": pd.Timestamp(row0["date"]),
        "cal_n_days": int(row0["n_days"]),
        "cal_window_start": pd.Timestamp(row0["window_start"]),
        "cal_window_end": pd.Timestamp(row0["window_end"]),
    }

# mc_params feed the interactive Monte Carlo in §5 (values appear after Start).


In [ ]:
# Rolling parameter paths (reference). Monte Carlo still uses fixed first-day values.
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for t in TICKERS:
    r = rolling[t]
    x = pd.to_datetime(r["date"])
    axes[0].plot(x, r["mu"], lw=1.2, label=t, color=COLORS[t])
    axes[1].plot(x, r["sigma"], lw=1.2, label=t, color=COLORS[t])
axes[0].axhline(0, color="0.5", lw=0.8)
axes[0].set_ylabel("μ̂ (annual)")
axes[0].set_title(f"Daily rolling drift ({LOOKBACK_DAYS}-day / 1y window)")
axes[0].legend(frameon=False, ncol=3)
axes[1].set_ylabel("σ̂ (annual)")
axes[1].set_title(f"Daily rolling volatility ({LOOKBACK_DAYS}-day / 1y window)")
axes[1].set_xlabel("Date")
axes[1].legend(frameon=False, ncol=3)
plt.tight_layout()
plt.show()


## 5. Interactive Monte Carlo (fixed parameters)

For **each** of AAPL, MSFT, and SPY:

| Graph | What you see |
|-------|----------------|
| **Left** | Monte Carlo paths + **one expected path** (mean across paths) |
| **Right** | That **expected path** vs **real historical** adjusted close over 2018–2019 |

Controls: **Start** runs a simulation; **Restart** draws a new seed and runs again.  
No sliders — \(\mu\), \(\sigma\), \(S_0\), \(T\), and step count are the fixed values from §4.


In [ ]:
def simulate_gbm(mu, sigma, S0, n_steps, n_paths, seed):
    """Daily-step GBM over n_steps intervals; returns paths shape (n_paths, n_steps+1)."""
    rng = np.random.default_rng(seed)
    dt = 1.0 / N_DAYS
    z = rng.standard_normal((n_paths, n_steps))
    increments = (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * z
    log_paths = np.cumsum(increments, axis=1)
    paths = S0 * np.exp(np.hstack([np.zeros((n_paths, 1)), log_paths]))
    return paths


def make_ticker_panel(ticker: str, n_paths: int = 80):
    p = mc_params[ticker]
    hist = period_prices[ticker].dropna()
    n_steps = len(hist) - 1
    dates = hist.index
    state = {"seed": 42}

    out = widgets.Output()
    btn_start = widgets.Button(description="Start", button_style="success", icon="play")
    btn_restart = widgets.Button(description="Restart", button_style="warning", icon="refresh")
    info = widgets.HTML(
        value=(
            f"<b>{ticker}</b> — fixed params: "
            f"μ̂={p['mu']:.4f}, σ̂={p['sigma']:.4f}, S₀={p['S0']:.4f}, "
            f"T≈{p['T_years']:.2f}y, steps={n_steps}, paths={n_paths} "
            f"(calibrated {p['cal_date'].date()}, {p['cal_n_days']}-day / 1y lookback)"
        )
    )

    def run(new_seed: bool = False):
        if new_seed:
            state["seed"] = int(np.random.default_rng().integers(0, 1_000_000_000))
        with out:
            clear_output(wait=True)
            paths = simulate_gbm(
                mu=p["mu"],
                sigma=p["sigma"],
                S0=p["S0"],
                n_steps=n_steps,
                n_paths=n_paths,
                seed=state["seed"],
            )
            expected = paths.mean(axis=0)

            fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))

            axes[0].plot(dates, paths.T, color=COLORS[ticker], alpha=0.12, lw=0.7)
            axes[0].plot(dates, expected, color="black", lw=2.2, label="expected path (MC mean)")
            axes[0].set_title(f"{ticker}: Monte Carlo (fixed GBM params)")
            axes[0].set_ylabel("price")
            axes[0].legend(loc="best", frameon=False)

            axes[1].plot(dates, hist.values, color=COLORS[ticker], lw=1.8, label="historical adj close")
            axes[1].plot(
                dates, expected, color="black", lw=2.0, ls="--", label="expected path (MC mean)"
            )
            axes[1].set_title(f"{ticker}: expected MC path vs history")
            axes[1].set_ylabel("price")
            axes[1].legend(loc="best", frameon=False)

            for ax in axes:
                ax.set_xlabel("date")
            fig.suptitle(
                f"{ticker}  |  seed={state['seed']}  |  μ={p['mu']:.4f}, σ={p['sigma']:.4f}",
                fontsize=11,
                y=1.02,
            )
            plt.tight_layout()
            plt.show()

            rmse = float(np.sqrt(np.mean((expected - hist.values) ** 2)))
            print(f"RMSE(expected vs historical) = {rmse:.4f}   |   seed = {state['seed']}")

    btn_start.on_click(lambda _: run(new_seed=False))
    btn_restart.on_click(lambda _: run(new_seed=True))

    return widgets.VBox([info, widgets.HBox([btn_start, btn_restart]), out])


display(Markdown("### Six interactive graphs — click **Start** (or **Restart**) for each ticker"))

for ticker in TICKERS:
    role = "primary" if ticker == "SPY" else "secondary"
    display(Markdown(f"#### {ticker} ({role})"))
    display(make_ticker_panel(ticker))


## 6. How to read the six graphs

1. **AAPL — Monte Carlo:** cloud of simulated paths; black line = expected trend (mean path).
2. **AAPL — expected vs history:** same black expected path against real 2018–2019 prices.
3. **MSFT — Monte Carlo**
4. **MSFT — expected vs history**
5. **SPY (primary) — Monte Carlo**
6. **SPY (primary) — expected vs history**

**Restart** only redraws randomness (new seed). Parameters stay locked to the §4 calibration.
